# Semana 5 — Conecta 4, minimax y poda α-β

**Plan de clase:** [`../weeks/week-05.md`](../weeks/week-05.md) · **Diapositivas:** [`../slides/week-05/deck.md`](../slides/week-05/deck.md)

Programación en vivo, 25 minutos. Construye el juego, después minimax y luego α-β,
registrando las medidas durante todo el proceso.

**Tres demostraciones:** la misma respuesta con menos nodos · ordenar las jugadas
aumenta la mejora · el efecto horizonte hace perder una partida delante de la clase.

El agente que construyas aquí participa en el **torneo de la sesión 5B**.


## 1 · El tablero

Conecta 4 de 7×6. Sin representación mediante mapas de bits y deliberadamente legible:
los estudiantes deben modificarlo en el taller, y a esta escala la claridad importa más que el ingenio.

`push` y `pop` modifican el tablero en el lugar, sin copiarlo. Esto importa: a profundidad 7,
copiar produciría cientos de miles de tableros y dominaría el tiempo de ejecución.


In [1]:
import time
from math import inf

# El tablero tiene 6 filas y 7 columnas; se gana alineando 4 fichas.
# Los enteros representan casillas: 0=vacía, MAX=1 y MIN=-1. Negar el jugador alterna el turno.
ROWS, COLS, WIN = 6, 7, 4
MAX, MIN = 1, -1        # MAX representa al jugador propio de la búsqueda


# POO: Connect4 es una clase, una definición de los datos y las operaciones de sus objetos.
# b = Connect4() crea una instancia: un tablero concreto e independiente de otros tableros.
# Los métodos consultan o modifican sus atributos; el código no define subclases propias.
class Connect4:
    # __init__ inicializa el objeto al crearlo. Python recibe automáticamente esa instancia como self.
    # self.grid, self.heights y self.last son atributos de INSTANCIA, no datos compartidos entre tableros.
    def __init__(self):
        # Cada fila es una lista nueva: la comprensión evita compartir la misma fila varias veces.
        # El índice de fila 0 corresponde a la parte inferior; las fichas se apilan hacia filas mayores.
        self.grid = [[0] * COLS for _ in range(ROWS)]
        self.heights = [0] * COLS      # cantidad de fichas en cada columna
        # Registra (fila, columna, jugador) de la última inserción; None significa que aún no hubo ninguna.
        self.last = None

    # ---- Mecánica del tablero ----------------------------------------
    # Método de consulta: b.legal_moves() usa b como self y devuelve las columnas que no están llenas.
    # La comprensión crea una lista nueva y no cambia el tablero.
    def legal_moves(self):
        return [c for c in range(COLS) if self.heights[c] < ROWS]

    # Método que modifica el MISMO objeto: coloca una ficha en la primera fila libre de la columna.
    # Supone una columna legal y un jugador MAX o MIN; no comprueba esos argumentos por sí mismo.
    def push(self, col, player):
        r = self.heights[col]
        self.grid[r][col] = player
        self.heights[col] += 1
        self.last = (r, col, player)

    # Deshace la ficha superior de una columna no vacía: restaura las casillas y su altura.
    # No restaura self.last; las funciones de búsqueda y evaluación de este notebook no lo consultan.
    def pop(self, col):
        self.heights[col] -= 1
        r = self.heights[col]
        self.grid[r][col] = 0

    # all devuelve True si todas las columnas tienen la altura máxima: ya no quedan jugadas legales.
    def full(self):
        return all(h == ROWS for h in self.heights)

    # ---- Comprobación de estados terminales --------------------------
    # El prefijo _ señala un auxiliar interno por convención, no una restricción de acceso de Python.
    # Desde (r,c), comprueba cuatro casillas en dirección (dr,dc), dentro del tablero y del mismo jugador.
    def _line(self, r, c, dr, dc):
        p = self.grid[r][c]
        if p == 0:
            return None
        for k in range(1, WIN):
            rr, cc = r + dr * k, c + dc * k
            if not (0 <= rr < ROWS and 0 <= cc < COLS) or self.grid[rr][cc] != p:
                return None
        return p

    # Prueba horizontal, vertical y ambas diagonales desde cada casilla ocupada.
    # Devuelve MAX o MIN si hay cuatro en línea; None indica que no encontró ganador.
    def winner(self):
        for r in range(ROWS):
            for c in range(COLS):
                if self.grid[r][c] == 0:
                    continue
                for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
                    w = self._line(r, c, dr, dc)
                    if w:
                        return w
        return None

    # ---- Evaluación --------------------------------------------------
    # yield convierte este método en un generador: produce un grupo de cuatro casillas por vez.
    # evaluate consume esos grupos sin crear primero una lista con todos ellos.
    def _windows(self):
        for r in range(ROWS):
            for c in range(COLS):
                for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
                    rr, cc = r + dr * (WIN - 1), c + dc * (WIN - 1)
                    if 0 <= rr < ROWS and 0 <= cc < COLS:
                        yield [self.grid[r + dr * k][c + dc * k] for k in range(WIN)]

    def evaluate(self):
        """Valor heurístico desde la perspectiva de MAX."""
        # Un método puede invocar otro sobre el mismo objeto: primero comprueba si ya existe un ganador.
        w = self.winner()
        if w == MAX:
            return 10_000
        if w == MIN:
            return -10_000
        score = 0
        # win es una lista de cuatro casillas. count es un método de list que cuenta fichas de cada jugador.
        for win in self._windows():
            mx, mn = win.count(MAX), win.count(MIN)
            if mx and mn:
                continue                       # grupo bloqueado: no aporta a ninguno de los dos jugadores
            # Los grupos de 1, 2 y 3 fichas valen 1, 10 y 50. Una línea ganadora de 4 se resolvió arriba.
            # Suma para MAX y resta para MIN: ambos jugadores usan la misma escala de evaluación.
            if mx:
                score += {1: 1, 2: 10, 3: 50}[mx]
            elif mn:
                score -= {1: 1, 2: 10, 3: 50}[mn]
        # El centro vale más: conocimiento clásico del dominio de Conecta 4.
        for r in range(ROWS):
            if self.grid[r][COLS // 2] == MAX:
                score += 3
            elif self.grid[r][COLS // 2] == MIN:
                score -= 3
        return score

    # Método especial: str(b) y print(b) lo invocan para obtener una representación legible del objeto.
    # La presentación invierte las filas para que la parte superior aparezca primero en pantalla.
    def __str__(self):
        sym = {0: ".", MAX: "X", MIN: "O"}
        rows = ["|" + " ".join(sym[self.grid[r][c]] for c in range(COLS)) + "|"
                for r in reversed(range(ROWS))]
        return "\n".join(rows) + "\n " + " ".join(str(c) for c in range(COLS))


# Instanciación y llamadas a métodos: b guarda un tablero; push modifica ese tablero y print usa __str__.
b = Connect4()
for col, p in [(3, MAX), (3, MIN), (4, MAX), (2, MIN)]:
    b.push(col, p)
print(b)
print("\nevaluate() =", b.evaluate(), " legal:", b.legal_moves())

|. . . . . . .|
|. . . . . . .|
|. . . . . . .|
|. . . . . . .|
|. . . O . . .|
|. . O X X . .|
 0 1 2 3 4 5 6

evaluate() = -5  legal: [0, 1, 2, 3, 4, 5, 6]


## 2 · Minimax

La recursión define el valor del juego: MAX maximiza, MIN minimiza,
y en ningún momento se supone que el oponente vaya a cometer errores.


In [2]:
# Función recursiva, no método de Connect4: board se recibe como argumento explícito.
# depth es la cantidad máxima de medias jugadas restantes; player indica quién decide en ese nivel.
# counter es una lista mutable compartida. El resultado es (valor desde MAX, columna elegida).
def minimax(board, depth, player, counter):
    # Cuenta todas las llamadas visitadas, incluidas hojas y estados terminales, no solo nodos con hijos generados.
    counter[0] += 1
    winner = board.winner()
    # Caso base: una victoria, el límite de búsqueda o un tablero lleno detienen esta rama.
    # Devuelve evaluate() y ninguna columna; el valor de una hoja no es una jugada.
    if winner is not None or depth == 0 or board.full():
        return board.evaluate(), None

    best_move = None
    if player == MAX:
        # MAX maximiza: -infinito permite que la primera jugada examinada sea mejor que el valor inicial.
        best = -inf
        for move in board.legal_moves():
            # Coloca una ficha en el objeto compartido y deja que el rival responda en la llamada recursiva.
            board.push(move, MAX)
            value, _ = minimax(board, depth - 1, MIN, counter)
            # Retroceso: deshace la ficha antes de explorar otra alternativa. No crea una copia del tablero.
            board.pop(move)
            # La comparación estricta conserva la primera columna encontrada si varias empatan en valor.
            if value > best:
                best, best_move = value, move
        return best, best_move
    else:
        # MIN minimiza la misma escala de MAX: comienza con +infinito.
        best = inf
        for move in board.legal_moves():
            # Ahora le toca a MIN; en el siguiente nivel decidirá MAX con una profundidad menos.
            board.push(move, MIN)
            value, _ = minimax(board, depth - 1, MAX, counter)
            # Retroceso: deshace la ficha antes de explorar otra alternativa. No crea una copia del tablero.
            board.pop(move)
            if value < best:
                best, best_move = value, move
        return best, best_move


board = Connect4()
board.push(3, MAX); board.push(3, MIN)
# Una lista permite que las llamadas recursivas actualicen el mismo contador visible desde fuera.
# Un entero local reasignado en cada llamada no compartiría esas actualizaciones de la misma forma.
c = [0]
# perf_counter mide segundos transcurridos; el contador de nodos mide trabajo del algoritmo.
t0 = time.perf_counter()
v, m = minimax(board, 4, MAX, c)
print(f"depth 4: value={v} move={m} nodes={c[0]:,} time={time.perf_counter()-t0:.3f}s")

depth 4: value=-4 move=3 nodes=2,801 time=0.109s


## 3 · Poda α-β

Un `break`: ahí se aplica toda la optimización.

`alpha` es el mejor valor que MAX ya puede garantizar; `beta`, el mejor que MIN
ya puede garantizar. Cuando se cruzan, el resto de esa rama no puede afectar
el resultado, así que dejamos de explorarla.


In [3]:
# Conserva el valor minimax de la raíz para el mismo horizonte y todas las alternativas legales,
# pero evita ramas que no pueden mejorar la decisión. En empates podría elegir otra columna.
# alpha y beta son cotas transmitidas por la recursión, no puntajes adicionales de evaluate().
def alphabeta(board, depth, player, alpha, beta, counter, order=None):
    # Misma convención de conteo que minimax: incluye también hojas y estados terminales.
    counter[0] += 1
    winner = board.winner()
    # Los mismos casos base y la misma evaluación que minimax permiten comparar sus valores.
    if winner is not None or depth == 0 or board.full():
        return board.evaluate(), None

    moves = board.legal_moves()
    # Usa la prioridad recibida y elimina columnas llenas. order debe contener TODAS las columnas legales:
    # una columna omitida no se explorará, por lo que no sería solo un cambio de orden.
    if order is not None:
        moves = [c for c in order if c in moves]

    best_move = None
    if player == MAX:
        best = -inf
        for move in moves:
            board.push(move, MAX)
            value, _ = alphabeta(board, depth - 1, MIN, alpha, beta, counter, order)
            # Restaura las casillas y alturas del tablero antes de evaluar otra jugada o aplicar una poda.
            board.pop(move)
            if value > best:
                best, best_move = value, move
            # MAX mejora su cota inferior con lo mejor que encontró en las alternativas examinadas.
            alpha = max(alpha, best)
            # Las cotas se cruzaron: un ancestro ya tiene una alternativa que hace irrelevante continuar esta rama.
            if alpha >= beta:
                break                      # aquí se aplica la poda
        return best, best_move
    else:
        best = inf
        for move in moves:
            board.push(move, MIN)
            value, _ = alphabeta(board, depth - 1, MAX, alpha, beta, counter, order)
            # Restaura las casillas y alturas del tablero antes de evaluar otra jugada o aplicar una poda.
            board.pop(move)
            if value < best:
                best, best_move = value, move
            # MIN reduce la cota superior: elige el valor más pequeño de la misma escala de MAX.
            beta = min(beta, best)
            # Las cotas se cruzaron: un ancestro ya tiene una alternativa que hace irrelevante continuar esta rama.
            if alpha >= beta:
                break
        return best, best_move

## 4 · Demostración 1 — La misma respuesta con muchos menos nodos

⚠️ **La poda no es una aproximación.** α-β devuelve el valor *idéntico* al de minimax.
Solo evita examinar ramas que, de forma demostrable, no pueden influir en la decisión.

La aserción de abajo es la línea importante de este notebook.


In [4]:
# Fábrica de una posición de prueba: devuelve una instancia nueva de Connect4 con dos fichas centrales.
# No es un método de clase; construye y devuelve un objeto para aislar cada búsqueda.
def fresh():
    b = Connect4()
    b.push(3, MAX); b.push(3, MIN)
    return b


print(f"  {'depth':>6}{'minimax':>12}{'alpha-beta':>13}{'speedup':>10}{'same value?':>13}")
print("  " + "-" * 55)
for depth in (2, 4, 5, 6):
    # Cada algoritmo recibe un tablero equivalente y su propio contador, para no mezclar sus resultados.
    b1, c1 = fresh(), [0]
    t0 = time.perf_counter()
    v1, _ = minimax(b1, depth, MAX, c1)
    t1 = time.perf_counter() - t0

    b2, c2 = fresh(), [0]
    t0 = time.perf_counter()
    # La ventana inicial (-infinito, +infinito) no restringe el valor de la raíz.
    # _ descarta la columna: aquí interesa comprobar el valor, no si ambos eligieron la misma jugada.
    v2, _ = alphabeta(b2, depth, MAX, -inf, inf, c2)
    t2 = time.perf_counter() - t0

    # assert lanza AssertionError si la poda cambia el valor: sería una diferencia de comportamiento.
    assert v1 == v2, f"VALUES DIFFER at depth {depth}: {v1} vs {v2}"
    # La razón c1/c2 compara nodos visitados, no tiempos. Las f-strings solo alinean la tabla.
    print(f"  {depth:>6}{c1[0]:>12,}{c2[0]:>13,}{c1[0]/c2[0]:>9.1f}x{'yes':>13}")

print("\nAll values identical. Pruning changed the COST, not the ANSWER.")

   depth     minimax   alpha-beta   speedup  same value?
  -------------------------------------------------------
       2          57           45      1.3x          yes
       4       2,801        1,002      2.8x          yes
       5      19,607        4,103      4.8x          yes
       6     132,179       17,030      7.8x          yes

All values identical. Pruning changed the COST, not the ANSWER.


## 5 · Demostración 2 — Ordenar las jugadas aumenta la mejora

La eficacia de α-β depende de examinar primero las buenas jugadas. En Conecta 4,
las columnas centrales son fuertes, así que `3, 2, 4, 1, 5, 0, 6` es un orden estático razonable.

Teoría: un orden perfecto da `b^(m/2)`, lo que **duplica la profundidad alcanzable**.
Un orden aleatorio da aproximadamente `b^(3m/4)`. Observa la columna central de la tabla.


In [5]:
# Listas estáticas de prioridad. Cada una incluye las siete columnas para no eliminar alternativas.
# WORST representa un orden que pospone el centro; se conserva aunque esta tabla no lo utilice.
CENTER_FIRST = [3, 2, 4, 1, 5, 0, 6]
LEFT_TO_RIGHT = [0, 1, 2, 3, 4, 5, 6]
WORST = [0, 6, 1, 5, 2, 4, 3]

print(f"  {'depth':>6}{'left-to-right':>15}{'center-first':>14}{'gain':>8}")
print("  " + "-" * 44)
for depth in (4, 5, 6, 7):
    # Cuenta alfa-beta con prioridad de izquierda a derecha y después con prioridad al centro.
    # fresh() entrega tableros equivalentes para comparar solo el efecto del orden.
    _, cl = None, [0]
    alphabeta(fresh(), depth, MAX, -inf, inf, cl, LEFT_TO_RIGHT)
    _, cc = None, [0]
    alphabeta(fresh(), depth, MAX, -inf, inf, cc, CENTER_FIRST)
    # El cociente muestra cuántas veces más nodos visitó el primer orden respecto al segundo.
    print(f"  {depth:>6}{cl[0]:>15,}{cc[0]:>14,}{cl[0]/cc[0]:>7.2f}x")

print("""
One line -- a sorted move list -- and the node count drops again. Ordering is not
a micro-optimization; it decides how deep you can afford to look, and depth is
strength. This is why real engines spend effort on move ordering before anything
else.""")

   depth  left-to-right  center-first    gain
  --------------------------------------------
       4          1,002           319   3.14x
       5          4,103         1,792   2.29x
       6         17,030         3,700   4.60x
       7         79,463        23,316   3.41x

One line -- a sorted move list -- and the node count drops again. Ordering is not
a micro-optimization; it decides how deep you can afford to look, and depth is
strength. This is why real engines spend effort on move ordering before anything
else.


## 6 · Demostración 3 — El efecto horizonte

La que los estudiantes recuerdan.

Un agente de **profundidad 4** no puede ver una amenaza que se resuelve en profundidad 5.
Hace una jugada que parece buena según su propia evaluación y pierde.
El problema no está en la función de evaluación, sino en el *horizonte*.


In [6]:
# Ejecuta una partida completa o hasta max_plies. Cada ply es una jugada de UN jugador.
# Esta función recibe profundidades fijas, no un presupuesto de tiempo ni un agente con profundización iterativa.
def play(depth_max, depth_min, order=CENTER_FIRST, max_plies=42, verbose=False):
    """Enfrenta MAX y MIN con alfa-beta a distintas profundidades. Devuelve (ganador, tablero)."""
    # Crea un objeto nuevo para esta partida; sus atributos no se comparten con otras partidas.
    board = Connect4()
    player = MAX
    for ply in range(max_plies):
        if board.winner() is not None or board.full():
            break
        # Escoge cuánto puede mirar el jugador actual. Ambos usan el mismo evaluate(), desde la perspectiva de MAX.
        d = depth_max if player == MAX else depth_min
        # Nueva búsqueda desde la posición actual; [0] crea un contador propio para esta decisión.
        # Las fichas hipotéticas se deshacen dentro de alfa-beta; aquí todavía no se hizo la jugada real.
        _, move = alphabeta(board, d, player, -inf, inf, [0], order)
        if move is None:
            break
        # Aplica la jugada REAL de la partida: esta ficha queda colocada para el siguiente turno.
        board.push(move, player)
        if verbose:
            print(f"ply {ply}: {'MAX' if player==MAX else 'MIN'} -> col {move}")
        # MAX=1 y MIN=-1: cambiar el signo alterna los turnos sin otra tabla de conversión.
        player = -player
    # Devuelve una tupla: el jugador ganador o None, y el objeto con el estado final.
    # Si se corta por max_plies, None también puede significar que no terminó: no se distingue aquí del empate.
    return board.winner(), board


# Compara partidas con distinta capacidad de anticipación y una con igual profundidad para ambos lados.
for dmax, dmin in [(2, 5), (4, 5), (5, 5), (6, 4)]:
    w, board = play(dmax, dmin)
    name = {MAX: "MAX wins", MIN: "MIN wins", None: "draw"}[w]
    print(f"MAX depth {dmax} vs MIN depth {dmin}:  {name}")

print("""
Depth is strength -- MOSTLY. See the note below on the 5-vs-5 row.
The shallow agent is not making
"mistakes" it could detect -- every move it plays is optimal WITH RESPECT TO ITS
HORIZON. It simply cannot see far enough.

That is the horizon effect, and no evaluation function fixes it. Only more depth
does -- which is exactly what alpha-beta buys you.""")

MAX depth 2 vs MIN depth 5:  MIN wins
MAX depth 4 vs MIN depth 5:  MIN wins
MAX depth 5 vs MIN depth 5:  MIN wins
MAX depth 6 vs MIN depth 4:  MAX wins

Depth is strength -- MOSTLY. See the note below on the 5-vs-5 row.
The shallow agent is not making
"mistakes" it could detect -- every move it plays is optimal WITH RESPECT TO ITS
HORIZON. It simply cannot see far enough.

That is the horizon effect, and no evaluation function fixes it. Only more depth
does -- which is exactly what alpha-beta buys you.


### ⚠️ Un resultado de esa tabla es un efecto particular de la implementación: indícalo

`MAX depth 5 vs MIN depth 5` → **gana MIN**. Con la *misma* profundidad ganó el segundo jugador.
Eso no es una propiedad de Conecta 4: con juego perfecto gana el **primero**.
Es un efecto particular de nuestra función de evaluación y de que MAX debe decidir
primero con información imperfecta.

De aquí se desprenden dos lecciones que se aplican a todo el curso:

1. **La profundidad aporta fuerza con respecto a una función de evaluación fija.**
   El resultado de 6 contra 4 es la señal relevante; el de 5 contra 5 informa sobre
   `evaluate()`, no sobre la búsqueda.
2. **No mantengas una explicación ordenada si tus resultados la contradicen.**
   La idea de que más profundidad siempre mejora es *mayormente* cierta aquí, y se ha señalado dónde no lo es.

Pide a la clase que explique la fila de 5 contra 5 antes de explicarla tú.
Permite detectar quién comprende la distinción entre evaluación y horizonte
mejor que cualquier pregunta de examen.


## 7 · Observa a un agente poco profundo perder una partida concreta

Imprime el tablero final de la partida entre profundidades 2 y 5.
Ese es el resultado que debes mostrar a la clase.


In [ ]:
# Desempaqueta el ganador y la instancia del tablero final de esta partida concreta.
w, board = play(2, 5)
# print invoca Connect4.__str__ para mostrar fichas X de MAX y O de MIN, con columnas numeradas 0 a 6.
print(board)
print(f"\nwinner: {'MAX (X)' if w == MAX else 'MIN (O)' if w == MIN else 'draw'}")
print("""
X is the depth-2 agent. Trace back and find the ply where it allowed a threat it
could not see. Every one of its moves was locally reasonable.""")

## 8 · Minimax supone un oponente perfecto

Conviene decirlo explícitamente porque esa suposición deja de cumplirse en el torneo:
minimax es **pesimista** y supone que MIN siempre elige la mejor respuesta.

Contra un oponente *débil*, esto es subóptimo: rechazará una jugada arriesgada con un beneficio
enorme que un oponente falible probablemente no sabría castigar.

> **Modelar al oponente como perfecto es seguro, no óptimo.** Pregunta qué buscas
> realmente en el torneo.


## 9 · Avance del taller

- Añade **tablas de transposiciones**: una misma posición alcanzada mediante órdenes
  de jugadas distintos se busca actualmente más de una vez.
- Añade **profundización iterativa** para respetar un presupuesto de *tiempo* en lugar
  de uno de profundidad. Eso es lo que exige el torneo.
- Después, la comparación de la semana: un **LLM que juega a Conecta 4** a partir de una
  descripción del tablero. Evalúa primero la legalidad y después la fuerza. Es de esperar
  que haya jugadas ilegales: deben registrarse en el Atlas de Fallos.

La distinción de Shannon de 1950 entre tipo A, fuerza bruta, y tipo B, selectivo, es un marco útil.
El LLM no pertenece a ninguno de los dos: es una **distribución previa aprendida sobre jugadas,
sin ninguna búsqueda**. Ruoss y colaboradores (2024) muestran que la planificación amortizada
puede alcanzar nivel de gran maestro de ajedrez sin buscar durante la inferencia.
La pregunta interesante no es «¿puede jugar?», sino **«¿adónde fue la búsqueda?»**
